In [ ]:
! pip install pandas requests python-dotenv openpyxl xlrd jupyterlab

In [ ]:
import pandas as pd
import numpy as np
import os

STAGING_DIR = "../data/staging/"

# Load EIA
eia_gen    = pd.read_csv(f"{STAGING_DIR}eia_generation_2022_2024.csv")
eia_retail = pd.read_csv(f"{STAGING_DIR}eia_retail_2022_2024.csv")

# Load World Bank
wb_cpi     = pd.read_csv(f"{STAGING_DIR}wb_cpi_2022_2024.csv")
wb_gdp     = pd.read_csv(f"{STAGING_DIR}wb_gdp_2022_2024.csv")
wb_ind     = pd.read_csv(f"{STAGING_DIR}wb_industrial_2022_2024.csv")
wb_unemp   = pd.read_csv(f"{STAGING_DIR}wb_unemployment_2022_2024.csv")
wb_fx      = pd.read_csv(f"{STAGING_DIR}wb_exchange_rate_2022_2024.csv")

print("Semua file berhasil di-load ✓")

In [ ]:
print("=" * 55)
print("SPARSITY — EIA DATASETS")
print("=" * 55)

for name, df in [("eia_generation", eia_gen), ("eia_retail", eia_retail)]:
    total_cells  = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    sparsity_pct  = (missing_cells / total_cells) * 100

    print(f"\n📄 {name}")
    print(f"   Shape           : {df.shape}")
    print(f"   Total cells     : {total_cells:,}")
    print(f"   Missing cells   : {missing_cells:,}")
    print(f"   Sparsity        : {sparsity_pct:.2f}%")
    print(f"\n   Missing per kolom:")
    missing_col = df.isnull().sum()
    missing_col = missing_col[missing_col > 0]
    if missing_col.empty:
        print("   → Tidak ada missing value")
    else:
        for col, cnt in missing_col.items():
            pct = cnt / len(df) * 100
            print(f"   → {col:<30} {cnt:>5} rows  ({pct:.1f}%)")

In [ ]:
print("=" * 55)
print("SPARSITY — WORLD BANK DATASETS")
print("=" * 55)

wb_datasets = {
    "wb_cpi"          : wb_cpi,
    "wb_gdp"          : wb_gdp,
    "wb_industrial"   : wb_ind,
    "wb_unemployment" : wb_unemp,
    "wb_exchange_rate": wb_fx,
}

for name, df in wb_datasets.items():
    total_cells   = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    sparsity_pct  = (missing_cells / total_cells) * 100

    print(f"\n📄 {name}")
    print(f"   Shape           : {df.shape}")
    print(f"   Total cells     : {total_cells:,}")
    print(f"   Missing cells   : {missing_cells:,}")
    print(f"   Sparsity        : {sparsity_pct:.2f}%")

    # Cek per negara
    if "country" in df.columns:
        value_col = [c for c in df.columns
                     if c not in ["period_raw", "country"]][0]
        missing_by_country = (
            df.groupby("country")[value_col]
            .apply(lambda x: x.isnull().sum())
        )
        missing_by_country = missing_by_country[missing_by_country > 0]
        if missing_by_country.empty:
            print("   → Semua negara lengkap, tidak ada missing")
        else:
            print(f"   Missing per negara (kolom: {value_col}):")
            for country, cnt in missing_by_country.items():
                pct = cnt / df[df["country"] == country].shape[0] * 100
                print(f"   → {country:<25} {cnt:>3} rows  ({pct:.0f}%)")

    # Coverage negara
    if "country" in df.columns:
        countries = sorted(df["country"].unique())
        print(f"   Negara tersedia : {countries}")

In [ ]:
print("=" * 55)
print("IMBALANCE — EIA GENERATION (distribusi fuel type)")
print("=" * 55)

# Identifikasi kolom fuel type dan generation
print("\nKolom EIA Generation:")
print(eia_gen.dtypes)
print()
print(eia_gen.head(3).to_string())

In [ ]:
# Sesuaikan nama kolom dengan output Cell 4
# Ganti 'fueltypeid' dan 'generation' jika nama kolomnya berbeda

fuel_col = None
gen_col  = None

# Auto-detect kolom fuel type
for c in eia_gen.columns:
    if "fuel" in c.lower() or "type" in c.lower():
        fuel_col = c
        break

# Auto-detect kolom generation
for c in eia_gen.columns:
    if "gen" in c.lower():
        gen_col = c
        break

print(f"Kolom fuel type terdeteksi : {fuel_col}")
print(f"Kolom generation terdeteksi: {gen_col}")

if fuel_col and gen_col:
    # Konversi ke numerik
    eia_gen[gen_col] = pd.to_numeric(eia_gen[gen_col], errors="coerce")

    gen_by_fuel = (
        eia_gen.groupby(fuel_col)[gen_col]
        .sum()
        .sort_values(ascending=False)
    )
    total_gen = gen_by_fuel.sum()

    print(f"\nTotal generasi seluruh fuel: {total_gen:,.0f} ribu MWh\n")
    print(f"{'Fuel Type':<20} {'Total (ribu MWh)':>20} {'Share (%)':>12} {'Ratio vs min':>14}")
    print("-" * 70)

    min_val = gen_by_fuel.min()
    for fuel, val in gen_by_fuel.items():
        share = val / total_gen * 100
        ratio = val / min_val if min_val > 0 else float("inf")
        print(f"{str(fuel):<20} {val:>20,.0f} {share:>11.2f}% {ratio:>13.1f}x")
else:
    print("⚠ Kolom fuel type atau generation tidak terdeteksi otomatis")
    print("  Sesuaikan nama kolom secara manual di cell ini")

In [ ]:
print("=" * 55)
print("IMBALANCE — WORLD BANK (distribusi nilai per negara)")
print("=" * 55)

for name, df in wb_datasets.items():
    if "country" not in df.columns:
        continue

    value_col = [c for c in df.columns
                 if c not in ["period_raw", "country"]][0]

    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

    stats = (
        df.groupby("country")[value_col]
        .mean()
        .sort_values(ascending=False)
    )

    min_val = stats.min()
    max_val = stats.max()
    ratio   = max_val / min_val if min_val > 0 else float("inf")

    print(f"\n📄 {name} (kolom: {value_col})")
    print(f"   Ratio max/min: {ratio:.1f}x")
    print(f"   {'Negara':<25} {'Rata-rata nilai':>18}")
    print("   " + "-" * 45)
    for country, val in stats.items():
        print(f"   {str(country):<25} {val:>18.4f}")

In [ ]:
print("=" * 55)
print("OUTLIER — EIA DATASETS (metode IQR)")
print("=" * 55)

def detect_outliers_iqr(df, col, label=""):
    series = pd.to_numeric(df[col], errors="coerce").dropna()
    if series.empty:
        return

    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = series[(series < lower) | (series > upper)]
    pct = len(outliers) / len(series) * 100

    print(f"\n   Kolom: {col}")
    print(f"   Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}")
    print(f"   Batas bawah: {lower:.2f}  |  Batas atas: {upper:.2f}")
    print(f"   Jumlah outlier: {len(outliers)} dari {len(series)} "
          f"({pct:.1f}%)")
    if not outliers.empty:
        print(f"   Nilai outlier  : min={outliers.min():.2f}, "
              f"max={outliers.max():.2f}")

# EIA Generation
print("\n📄 eia_generation")
numeric_cols_gen = eia_gen.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols_gen:
    detect_outliers_iqr(eia_gen, col)

# EIA Retail
print("\n📄 eia_retail")
numeric_cols_ret = eia_retail.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols_ret:
    detect_outliers_iqr(eia_retail, col)

In [ ]:
print("=" * 55)
print("OUTLIER — WORLD BANK DATASETS (metode IQR)")
print("=" * 55)

for name, df in wb_datasets.items():
    print(f"\n📄 {name}")
    value_col = [c for c in df.columns
                 if c not in ["period_raw", "country"]][0]
    detect_outliers_iqr(df, value_col)

    # Outlier breakdown per negara
    print(f"   Breakdown outlier per negara:")
    for country in sorted(df["country"].unique()):
        subset = df[df["country"] == country].copy()
        detect_outliers_iqr(subset, value_col,
                             label=f"  [{country}]")

In [ ]:
print("=" * 55)
print("RINGKASAN DATA QUALITY REPORT")
print("=" * 55)

all_datasets = {
    "eia_generation"  : eia_gen,
    "eia_retail"      : eia_retail,
    "wb_cpi"          : wb_cpi,
    "wb_gdp"          : wb_gdp,
    "wb_industrial"   : wb_ind,
    "wb_unemployment" : wb_unemp,
    "wb_exchange_rate": wb_fx,
}

print(f"\n{'Dataset':<22} {'Shape':>12} {'Sparsity':>10} "
      f"{'Missing':>10} {'Numerik cols':>14}")
print("-" * 72)

for name, df in all_datasets.items():
    total    = df.shape[0] * df.shape[1]
    missing  = df.isnull().sum().sum()
    sparsity = missing / total * 100
    num_cols = df.select_dtypes(include=[np.number]).shape[1]
    print(f"{name:<22} {str(df.shape):>12} {sparsity:>9.2f}% "
          f"{missing:>10,} {num_cols:>14}")

print("\nCatatan:")
print("- Sparsity dihitung dari total missing cells / total cells")
print("- Outlier dideteksi dengan metode IQR (1.5 × IQR rule)")
print("- Imbalance dinilai dari ratio max/min distribusi nilai")